# MARICL-AL on ACTG-175 (Semi-Synthetic)

## Dataset

- **Source**: AIDS Clinical Trials Group Study 175 (Hammer et al., 1996). Real covariate structure, synthetic outcomes.
- **Original n**: 2139 patients.
- **Treatment**: `treat` column (binary: 0 = ZDV only, 1 = other/combination therapy).
- **12 covariates**: `age, wtkg, hemo, homo, drugs, karnof, race, gender, z30, cd40, cd80, str2`

## Outcome DGP (from R-Design)

Formulas used for synthetic targets:

```
µ(x,0) = fMLP(x) + 6 + 0.3·wtkg² - sin(age)·(gender+1) + 0.6·hemo·race - 0.2·z30
τ(x)   = 1.5·sin(wtkg)·(karnof_hi+1) + 2·age
µ(x,1) = µ(x,0) + τ(x)
```

## Implementation Assumptions

| Aspect | Assumption |
|---|---|
| 12 covariates | `age, wtkg, hemo, homo, drugs, karnof, race, gender, z30, cd40, cd80, str2` |
| `fMLP` architecture | 2-layer numpy MLP (12→16→8→1), tanh activations, seed=0 |
| `karnof_hi` | `(karnof >= 90).astype(float)` |
| Age rejection sampling | P(reject) = 0.7 if age > median, else 0 |

## 1. Setup


In [ ]:
# Hyperparams
num_hypothesis = 2
random_seed = 4
llm_seed = random_seed
data_seed = random_seed
TREATMENT_IDX = 12  # ACTG-175: treatment appended at index 12

In [ ]:
# !pip install -q langchain langchain-openai openai pydantic python-dotenv numpy scipy scikit-learn xgboost matplotlib pandas


In [ ]:
import os, ast, math, json, time, warnings
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional, Callable, Any
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C, WhiteKernel
import xgboost as xgb

from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import SystemMessage, HumanMessage

warnings.filterwarnings("ignore")
np.random.seed(0)

try:
    from dotenv import load_dotenv

    load_dotenv(Path.cwd().resolve() / ".env")
except ImportError:
    pass
_key = (os.environ.get("OPENAI_API_KEY") or os.environ.get("API_KEY", "")).strip()
if _key:
    os.environ["OPENAI_API_KEY"] = _key
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY"

os.environ["LANGCHAIN_TRACING_V2"] = "false"
os.environ["LANGCHAIN_CALLBACKS_BACKGROUND"] = "false"

LLM_MODEL = os.environ.get("MAICL_MODEL_NAME", "gpt-4o-mini")
EMB_MODEL = "text-embedding-3-small"
_re = (os.environ.get("OPENAI_REASONING_EFFORT", "low") or "").strip().lower()
REASONING_EFFORT = None if _re in ("", "none", "off") else _re
_kw = dict(model=LLM_MODEL, temperature=0.2, model_kwargs={"seed": llm_seed})
_kwc = dict(model=LLM_MODEL, temperature=0.7, model_kwargs={"seed": llm_seed})
if REASONING_EFFORT and LLM_MODEL.startswith(("o", "gpt-5")):
    _kw["reasoning_effort"] = REASONING_EFFORT
    _kwc["reasoning_effort"] = REASONING_EFFORT
llm = ChatOpenAI(**_kw)
llm_creative = ChatOpenAI(**_kwc)
embedder = OpenAIEmbeddings(model=EMB_MODEL)
print(f"env ready  ({LLM_MODEL}, {EMB_MODEL})")

## 2. MARICL core - **unchanged from v1 AL**

In [ ]:
def _safe_exp(x):
    return np.exp(np.clip(np.asarray(x, dtype=float), -50, 50))


def _safe_log(x):
    return np.log(np.maximum(np.asarray(x, dtype=float), 1e-10))


def _safe_sqrt(x):
    return np.sqrt(np.maximum(np.asarray(x, dtype=float), 0.0))


def _sigmoid(x):
    return 1.0 / (1.0 + _safe_exp(-np.asarray(x, dtype=float)))


def _mm(x, K):
    x = np.asarray(x, dtype=float)
    K = np.asarray(K, dtype=float)
    return x / (np.maximum(np.abs(K), 1e-9) + np.abs(x) + 1e-9)


def _gauss(x, mu, sigma):
    x = np.asarray(x, dtype=float)
    mu = np.asarray(mu, dtype=float)
    sigma = np.asarray(sigma, dtype=float)
    return _safe_exp(-((x - mu) ** 2) / (2.0 * np.maximum(sigma**2, 1e-9)))


def _relu(x):
    return np.maximum(np.asarray(x, dtype=float), 0.0)


ALLOWED_FUNCS: Dict[str, Callable] = {
    "clip": np.clip,
    "exp": _safe_exp,
    "log": _safe_log,
    "log1p": np.log1p,
    "maximum": np.maximum,
    "minimum": np.minimum,
    "abs": np.abs,
    "sqrt": _safe_sqrt,
    "sigmoid": _sigmoid,
    "tanh": np.tanh,
    "where": np.where,
    "sign": np.sign,
    "mm": _mm,
    "gauss": _gauss,
    "relu": _relu,
    "pi": np.pi,
    "e": np.e,
}
_ALLOWED_NODES = (
    ast.Expression,
    ast.Load,
    ast.Name,
    ast.Constant,
    ast.Num,
    ast.BinOp,
    ast.UnaryOp,
    ast.BoolOp,
    ast.Compare,
    ast.IfExp,
    ast.Call,
    ast.Tuple,
    ast.List,
    ast.Add,
    ast.Sub,
    ast.Mult,
    ast.Div,
    ast.Pow,
    ast.Mod,
    ast.USub,
    ast.UAdd,
    ast.And,
    ast.Or,
    ast.Not,
    ast.Eq,
    ast.NotEq,
    ast.Lt,
    ast.LtE,
    ast.Gt,
    ast.GtE,
)


def validate_formula(expr, allowed_features):
    if not expr or len(expr) > 2000:
        return False, "empty / too long"
    try:
        tree = ast.parse(expr, mode="eval")
    except SyntaxError as e:
        return False, f"syntax: {e}"
    allowed = set(ALLOWED_FUNCS) | set(allowed_features)
    for node in ast.walk(tree):
        if not isinstance(node, _ALLOWED_NODES):
            return False, f"disallowed node: {type(node).__name__}"
        if isinstance(node, ast.Call):
            if not isinstance(node.func, ast.Name):
                return False, "call must be bare name"
            if node.func.id not in ALLOWED_FUNCS:
                return False, f"bad func {node.func.id}"
        if isinstance(node, ast.Name):
            if node.id.startswith("_"):
                return False, f"private: {node.id}"
            if node.id not in allowed:
                return False, f"unknown name: {node.id}"
    return True, "ok"


def execute_formula(expr, X, feature_names, output_range=None):
    ok, why = validate_formula(expr, feature_names)
    if not ok:
        raise ValueError(f"validation: {why}")
    X = np.asarray(X, dtype=float)
    if X.ndim == 1:
        X = X.reshape(1, -1)
    ns = {**ALLOWED_FUNCS, **{n: X[:, i] for i, n in enumerate(feature_names)}}
    try:
        out = eval(expr, {"__builtins__": {}}, ns)
    except Exception as e:
        raise ValueError(f"runtime: {e}")
    out = np.asarray(out, dtype=float)
    if out.ndim == 0:
        out = np.full(X.shape[0], float(out))
    if not np.all(np.isfinite(out)):
        raise ValueError("non-finite")
    if output_range is not None:
        out = np.clip(out, *output_range)
    return out

In [ ]:
class Hypothesis(BaseModel):
    hypothesised_pattern: str
    implicated_features: List[str]
    functional_form_guess: str
    rationale: str


class Correction(BaseModel):
    T_k: str
    f_k: str


class Critique(BaseModel):
    diagnosis: str
    structural_or_coefficient: str
    suggested_change: str


ENCODER_PROMPTS = {
    "error_patterns": (
        "You are diagnosing systematic errors of a base ML model on an HIV treatment "
        "effect estimation task (ACTG-175). Focus on which covariate combinations "
        "(patient age, weight, Karnofsky score, CD4/CD8 counts, treatment assignment) "
        "drive large residuals, and what nonlinear shapes (interaction between weight and "
        "Karnofsky status, age-modulated treatment response, CD4 thresholds) the base "
        "model fails to capture."
    ),
    "sample_patterns": (
        "You are inferring covariate-outcome relationships from high-residual examples "
        "in the ACTG-175 HIV clinical trial. Focus on: heterogeneous treatment effects "
        "by patient weight and functional status (karnof), age modifying treatment "
        "response, and interaction effects between treatment (T) and baseline CD4 count."
    ),
}


def format_high_res_table(X, y, base_pred, r, fn, max_rows=15):
    idx = np.argsort(-np.abs(r))[:max_rows]
    header = "  ".join(
        ["idx"]
        + [f"{n:>10s}" for n in fn]
        + [f"{c:>10s}" for c in ["target", "base_pred", "residual"]]
    )
    lines = [header]
    for i in idx:
        row = [f"{i:3d}"] + [f"{X[i,j]:+10.3f}" for j in range(X.shape[1])]
        row += [
            f"{float(y[i]):+10.3f}",
            f"{float(base_pred[i]):+10.3f}",
            f"{float(r[i]):+10.3f}",
        ]
        lines.append("  ".join(row))
    return "\n".join(lines)


def build_aug_context(X_hr, y_hr, base_hr, r_hr, fn, fd, dc):
    feat_block = "\n".join(f"  - {n}: {d}" for n, d in zip(fn, fd))
    return (
        "DOMAIN: "
        + dc
        + "\n\nFEATURES:\n"
        + feat_block
        + "\n\nHIGH-RESIDUAL EXAMPLES:\n"
        + format_high_res_table(X_hr, y_hr, base_hr, r_hr, fn)
    )


def encode_hypothesis(aug_ctx, prompt_variant):
    sys_msg = (
        "You are an expert biostatistician + ML diagnostician. Given evidence "
        "of where a base model fails on HIV treatment outcome prediction (ACTG-175), "
        "hypothesise the missing mechanism — heterogeneous treatment effect, "
        "covariate interaction, or nonlinearity — as a closed-form correction."
    )
    user = ENCODER_PROMPTS[prompt_variant] + "\n\n" + aug_ctx
    return llm_creative.with_structured_output(Hypothesis).invoke(
        [SystemMessage(content=sys_msg), HumanMessage(content=user)]
    )


DECODER_SYS = (
    "Convert a structured hypothesis about HIV treatment outcome residuals into an "
    "executable correction. Context: ACTG-175 — predicting cognitive/functional outcomes "
    "from patient covariates and binary treatment T (0=ZDV only, 1=combination/other).\n"
    "Rules:\n"
    "1) f_k MUST be a single Python expression (no def, lambda, statements).\n"
    "2) Allowed names: feature names + {clip, exp, log, log1p, maximum, minimum, abs, sqrt, "
    "sigmoid, tanh, where, sign, mm, gauss, relu} + constants pi, e.\n"
    "3) Use sigmoid for threshold effects (karnof, age); gauss for continuous optima. "
    "Consider T * f(covariates) for heterogeneous treatment effect corrections.\n"
    "4) Add small constants to denominators; bound exp args; keep coefficients moderate.\n"
    "5) Features are standardized; output should be a scalar outcome correction."
)


def decode_correction(hyp, feature_names, output_range, prior_state=""):
    user = (
        f"FEATURES: {feature_names}\nOUTPUT_RANGE: {output_range}\n\n"
        f"HYPOTHESIS:\n  pattern: {hyp.hypothesised_pattern}\n"
        f"  features: {hyp.implicated_features}\n"
        f"  form: {hyp.functional_form_guess}\n"
        f"  rationale: {hyp.rationale}\n"
    )
    if prior_state:
        user += "\nPRIOR ITERATIONS:\n" + prior_state
    user += "\nProduce T_k (2-3 sentences) and f_k (single Python expression)."
    return llm.with_structured_output(Correction).invoke(
        [SystemMessage(content=DECODER_SYS), HumanMessage(content=user)]
    )


def format_failure_table(X_f, y_f, pred_f, err_f, fn, max_rows=10):
    idx = np.argsort(-np.abs(err_f))[:max_rows]
    header = "  ".join(
        [f"{n:>10s}" for n in fn] + [f"{c:>10s}" for c in ["target", "pred", "error"]]
    )
    lines = [header]
    for i in idx:
        row = [f"{X_f[i,j]:+10.3f}" for j in range(X_f.shape[1])]
        row += [
            f"{float(y_f[i]):+10.3f}",
            f"{float(pred_f[i]):+10.3f}",
            f"{float(err_f[i]):+10.3f}",
        ]
        lines.append("  ".join(row))
    return "\n".join(lines)


CRITIQUE_SYS = (
    "You diagnose why a candidate correction fails on the hardest remaining ACTG-175 "
    "examples. Distinguish structural mismatch (wrong interaction shape) from "
    "coefficient mismatch. Consider whether treatment indicator T is properly "
    "incorporated and whether age/weight/karnof interactions are captured."
)


def critique_correction(hyp, corr, loss, ft):
    user = (
        f"HYPOTHESIS: {hyp.hypothesised_pattern}\nCURRENT_T_k: {corr.T_k}\n"
        f"CURRENT_f_k: {corr.f_k}\nTRAIN_MAE: {loss:.4f}\n\n"
        f"WORST FAILURES:\n{ft}\n\nDiagnose and suggest a concrete refinement."
    )
    return llm.with_structured_output(Critique).invoke(
        [SystemMessage(content=CRITIQUE_SYS), HumanMessage(content=user)]
    )

In [ ]:
@dataclass
class CorrectionState:
    k: int
    hypothesis: Hypothesis
    history: List[Dict[str, Any]] = field(default_factory=list)
    best_loss: float = float("inf")
    best_correction: Optional[Correction] = None
    best_t: int = -1

    def add(self, t, correction, loss, critique=None):
        self.history.append(
            dict(t=t, correction=correction, loss=loss, critique=critique)
        )
        if loss < self.best_loss:
            self.best_loss = loss
            self.best_correction = correction
            self.best_t = t

    def prior_state_str(self):
        if not self.history:
            return ""
        ls = []
        for e in self.history[-3:]:
            ls.append(f"  iter {e['t']}: loss={e['loss']:.4f}")
            ls.append(f"    f_k: {e['correction'].f_k}")
            if e["critique"]:
                ls.append(f"    diagnosis: {e['critique'].diagnosis}")
                ls.append(f"    suggestion: {e['critique'].suggested_change}")
        return "\n".join(ls)


class MARICL:
    """Verbatim from v1 AL."""

    def __init__(
        self,
        K=2,
        kappa=0.3,
        T=3,
        B=10,
        p_min=0.1,
        gamma=2.0,
        tau_fail=0.1,
        tau_reg=0.2,
        output_range=(-0.5, 0.5),
        verbose=True,
    ):
        self.K = K
        self.kappa = kappa
        self.T = T
        self.B = B
        self.p_min = p_min
        self.gamma = gamma
        self.tau_fail = tau_fail
        self.tau_reg = tau_reg
        self.output_range = output_range
        self.verbose = verbose

    def _log(self, *a):
        if self.verbose:
            print(*a)

    def _apply(self, corr, X):
        return execute_formula(
            corr.f_k, X, self.feature_names, output_range=self.output_range
        )

    def _loss(self, corr, X, y, base_pred):
        try:
            pred = base_pred + self._apply(corr, X)
        except Exception:
            return float("inf"), None
        return float(mean_absolute_error(y, pred)), pred

    def _failure(self, X, y, cur_pred):
        err = cur_pred - y
        return np.abs(err) > self.tau_fail, err

    def _fit_conf_stats(self, X_hr_std):
        if len(X_hr_std) < 2:
            self._D95 = 1.0
            self._X_hr_std = X_hr_std
            return
        d = np.linalg.norm(X_hr_std[:, None, :] - X_hr_std[None, :, :], axis=-1)
        iu = np.triu_indices_from(d, k=1)
        self._D95 = max(float(np.percentile(d[iu], 95)), 1e-8)
        self._X_hr_std = X_hr_std

    def _confidence(self, X_std):
        d_t = np.linalg.norm(
            X_std[:, None, :] - self._X_hr_std[None, :, :], axis=-1
        ).min(axis=1)
        return _sigmoid(self.gamma * (1.0 - np.minimum(d_t / self._D95, 1.0)))

    def _batched_encode(self, X_hr, y_hr, base_hr, r_hr, fn, fd, dc, variant):
        N = len(X_hr)
        if N <= self.B:
            ctx = build_aug_context(X_hr, y_hr, base_hr, r_hr, fn, fd, dc)
            return encode_hypothesis(ctx, variant)
        shards = []
        for s in range(0, N, self.B):
            e = min(s + self.B, N)
            ctx = build_aug_context(
                X_hr[s:e], y_hr[s:e], base_hr[s:e], r_hr[s:e], fn, fd, dc
            )
            shards.append(encode_hypothesis(ctx, variant))
        return Hypothesis(
            hypothesised_pattern=shards[0].hypothesised_pattern,
            implicated_features=list(
                dict.fromkeys(f for s in shards for f in s.implicated_features)
            ),
            functional_form_guess=" || ".join(s.functional_form_guess for s in shards),
            rationale=" | ".join(s.rationale for s in shards),
        )

    def fit(self, X, y, base_model, fn, fd, dc):
        self.feature_names = list(fn)
        self.base_model = base_model
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        self._mu = X.mean(0)
        self._sd = X.std(0)
        self._sd[self._sd < 1e-9] = 1.0
        X_std = (X - self._mu) / self._sd
        base_pred = base_model.predict(X)
        r = y - base_pred
        n_top = max(2, int(self.kappa * len(y)))
        hr_idx = np.argsort(-np.abs(r))[:n_top]
        self._log(f"  high-residual pool: {len(hr_idx)} / {len(y)}")
        X_hr, y_hr = X[hr_idx], y[hr_idx]
        base_hr = base_pred[hr_idx]
        r_hr = r[hr_idx]
        self._fit_conf_stats(X_std[hr_idx])
        variants = list(ENCODER_PROMPTS.keys())
        variants = [variants[i % len(variants)] for i in range(self.K)]
        states = []
        for k in range(self.K):
            self._log(f"  [corr {k}] encoder={variants[k]}")
            z = self._batched_encode(
                X_hr, y_hr, base_hr, r_hr, self.feature_names, fd, dc, variants[k]
            )
            self._log(f"    hypothesis: {z.hypothesised_pattern}")
            self._log(f"    form guess: {z.functional_form_guess[:150]}")
            corr = None
            for _ in range(3):
                try:
                    corr = decode_correction(z, self.feature_names, self.output_range)
                    if not validate_formula(corr.f_k, self.feature_names)[0]:
                        corr = None
                        continue
                    _ = self._apply(corr, X[:5])
                    break
                except Exception:
                    corr = None
            if corr is None:
                corr = Correction(T_k="(none)", f_k="0.0")
            l, _ = self._loss(corr, X, y, base_pred)
            self._log(f"    init f_k: {corr.f_k[:120]}   loss={l:.4f}")
            s = CorrectionState(k=k, hypothesis=z)
            s.add(0, corr, l)
            states.append(s)
        for k in range(self.K):
            s = states[k]
            for t in range(self.T):
                cur = s.history[-1]["correction"]
                cur_l = s.history[-1]["loss"]
                pred_ens = base_pred.copy()
                for ss in states:
                    try:
                        pred_ens = pred_ens + 0.5 * self._apply(
                            ss.history[-1]["correction"], X
                        )
                    except Exception:
                        pass
                fmask, err = self._failure(X, y, pred_ens)
                if fmask.sum() == 0:
                    break
                ft = format_failure_table(
                    X[fmask], y[fmask], pred_ens[fmask], err[fmask], self.feature_names
                )
                try:
                    g = critique_correction(s.hypothesis, cur, cur_l, ft)
                except Exception:
                    break
                prior = (
                    s.prior_state_str()
                    + f"\n  critique: {g.diagnosis}\n  suggestion: {g.suggested_change}"
                )
                new = None
                for _ in range(3):
                    try:
                        new = decode_correction(
                            s.hypothesis, self.feature_names, self.output_range, prior
                        )
                        if not validate_formula(new.f_k, self.feature_names)[0]:
                            new = None
                            continue
                        _ = self._apply(new, X[:5])
                        break
                    except Exception:
                        new = None
                if new is None:
                    break
                new_l, _ = self._loss(new, X, y, base_pred)
                self._log(f"  [corr {k}] iter {t+1}: {cur_l:.4f} -> {new_l:.4f}")
                s.add(t + 1, new, new_l, g)
            self._log(f"  [corr {k}] best loss={s.best_loss:.4f} at t={s.best_t}")
        self.states_ = states
        self.scores_ = []
        self.bar_f_ = []
        for s in self.states_:
            if s.best_correction is None:
                self.scores_.append(0.0)
                self.bar_f_.append(0.0)
                continue
            try:
                d = self._apply(s.best_correction, X)
                mae = mean_absolute_error(y, base_pred + d)
                self.scores_.append(float(np.exp(-mae / max(self.tau_reg, 1e-9))))
                self.bar_f_.append(float(d.mean()))
            except Exception:
                self.scores_.append(0.0)
                self.bar_f_.append(0.0)
        self._log(f"  p_k = {[f'{p:.3f}' for p in self.scores_]}")
        self._log("  f_k templates:")
        for k, s in enumerate(self.states_):
            self._log(
                f"    [{k}] {s.best_correction.T_k if s.best_correction is not None else '(none)'}"
            )
        return self

    def predict_components(self, X):
        X = np.asarray(X, dtype=float)
        if X.ndim == 1:
            X = X.reshape(1, -1)
        X_std = (X - self._mu) / self._sd
        base = self.base_model.predict(X)
        deltas, ps, bars = [], [], []
        for s, p, bar in zip(self.states_, self.scores_, self.bar_f_):
            if s.best_correction is None or p <= self.p_min:
                continue
            try:
                deltas.append(self._apply(s.best_correction, X))
                ps.append(p)
                bars.append(bar)
            except Exception:
                continue
        if not deltas:
            return (
                base,
                np.zeros((X.shape[0], 0)),
                np.array([]),
                np.array([]),
                np.zeros(X.shape[0]),
            )
        c_x = self._confidence(X_std)
        stacked = np.stack(deltas, axis=1)
        p_arr = np.asarray(ps)
        bars = np.asarray(bars)
        w = p_arr[None, :] * c_x[:, None]
        Z = w.sum(1, keepdims=True)
        alpha = np.where(Z > 0, w / np.where(Z > 0, Z, 1.0), 0.0)
        return base, stacked, alpha, p_arr, bars

    def predict(self, X):
        base, deltas, alpha, p_arr, bars = self.predict_components(X)
        if deltas.shape[1] == 0:
            return base
        return base + (alpha * deltas).sum(axis=1)

## 3. ACTG-175 dataset and synthetic DGP


In [ ]:
CSV_PATH = "AIDS_ClinicalTrial_GroupStudy175.csv"

df = pd.read_csv(CSV_PATH)
print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f'Treatment distribution: {df["treat"].value_counts().to_dict()}')

# ── 12 covariates (assumption: see header) ─────────────────────────────────
ACTG_FEATURE_COLS = [
    "age",
    "wtkg",
    "hemo",
    "homo",
    "drugs",
    "karnof",
    "race",
    "gender",
    "z30",
    "cd40",
    "cd80",
    "str2",
]
X_raw = df[ACTG_FEATURE_COLS].values.astype(float)
t_raw = df["treat"].values.astype(float)  # binary: 0=ZDV, 1=other
n_total = len(X_raw)

# Standardize all features (including binary — consistent with IHDP approach)
scaler_actg = StandardScaler()
X_std_all = scaler_actg.fit_transform(X_raw)

# ── Derived feature: karnof_hi
karnof_hi = (X_raw[:, 5] >= 90).astype(float)  # index 5 = karnof

# Shorthand for standardized columns used in DGP
age_s = X_std_all[:, 0]  # age (standardized)
wtkg_s = X_std_all[:, 1]  # wtkg (standardized)
hemo_s = X_raw[:, 2]  # hemo (binary, raw)
race_s = X_raw[:, 6]  # race (binary, raw)
gender_s = X_raw[:, 7]  # gender (binary, raw)
z30_s = X_raw[:, 8]  # z30 (binary, raw)


# ── fMLP: small 2-layer numpy MLP with fixed seed
def make_mlp_numpy(input_dim=12, h1=16, h2=8, seed=0):
    rng = np.random.default_rng(seed)
    W1 = rng.normal(0, 0.5, (h1, input_dim))
    b1 = rng.normal(0, 0.1, h1)
    W2 = rng.normal(0, 0.5, (h2, h1))
    b2 = rng.normal(0, 0.1, h2)
    W3 = rng.normal(0, 0.5, (1, h2))
    b3 = rng.normal(0, 0.1, 1)

    def forward(X):
        h = np.tanh(X @ W1.T + b1)
        h = np.tanh(h @ W2.T + b2)
        return (h @ W3.T + b3).ravel()

    return forward


f_mlp = make_mlp_numpy(input_dim=12, seed=0)

# ── Synthetic potential outcomes (R-Design Appendix B.3.3) ─────────────────
# µ(x,0) = fMLP(x) + 6 + 0.3·wtkg² - sin(age)·(gender+1) + 0.6·hemo·race - 0.2·z30
mu0_full = (
    f_mlp(X_std_all)
    + 6
    + 0.3 * wtkg_s**2
    - np.sin(age_s) * (gender_s + 1)
    + 0.6 * hemo_s * race_s
    - 0.2 * z30_s
)

# τ(x) = 1.5·sin(wtkg)·(karnof_hi+1) + 2·age
tau_full = 1.5 * np.sin(wtkg_s) * (karnof_hi + 1) + 2 * age_s
mu1_full = mu0_full + tau_full

# ── Confounding bias η(x) (same form as IHDP, γ=1.5, seed=42) ─────────────
rng_conf = np.random.default_rng(42)
c1, c2, c3 = rng_conf.uniform(-1, 1, 3)
gamma = 1.5
eta_full = gamma * (1 + c1 * age_s + c2 * wtkg_s + c3 * age_s**2)
print(f"Confounding coefficients: c1={c1:.3f}, c2={c2:.3f}, c3={c3:.3f}")

# ── Data split 4:1:2 (obs_train / obs_test / rct_pool) ────────────────────
rng_split = np.random.default_rng(0)
idx_all = rng_split.permutation(n_total)
n_rct = int(n_total * 2 / 7)
n_obs_test = int(n_total * 1 / 7)
n_obs_train = n_total - n_rct - n_obs_test

obs_train_idx = idx_all[:n_obs_train]
obs_test_idx = idx_all[n_obs_train : n_obs_train + n_obs_test]
rct_all_idx = idx_all[n_obs_train + n_obs_test :]

# ── Age-based covariate shift in RCT pool (assumption: P(reject|old)=0.7) ─
median_age = np.median(X_raw[:, 0])
rng_age = np.random.default_rng(1)
age_rct = X_raw[rct_all_idx, 0]
reject_prob = np.where(age_rct > median_age, 0.7, 0.0)
keep_mask = rng_age.random(len(rct_all_idx)) > reject_prob
rct_idx = rct_all_idx[keep_mask]
print(f"RCT pool after age shift: {keep_mask.sum()} / {len(rct_all_idx)} retained")

# ── Generate observed outcomes ─────────────────────────────────────────────
rng_noise = np.random.default_rng(2)

# Observational train: confounded outcomes
y_obs_train = (
    mu0_full[obs_train_idx]
    + t_raw[obs_train_idx] * tau_full[obs_train_idx]
    + (2 * t_raw[obs_train_idx] - 1) * eta_full[obs_train_idx]
    + rng_noise.normal(0, 1, len(obs_train_idx))
)

# RCT pool: unconfounded factual outcomes
y_rct = (
    mu0_full[rct_idx]
    + t_raw[rct_idx] * tau_full[rct_idx]
    + rng_noise.normal(0, 1, len(rct_idx))
)

# Test: ground-truth µ0, µ1 used for PEHE (no noise needed)
mu0_test_actg = mu0_full[obs_test_idx]
mu1_test_actg = mu1_full[obs_test_idx]

# Feature arrays with treatment appended
X_obs_train = np.hstack([X_std_all[obs_train_idx], t_raw[obs_train_idx, None]])
X_rct = np.hstack([X_std_all[rct_idx], t_raw[rct_idx, None]])
X_test_actg = np.hstack([X_std_all[obs_test_idx], t_raw[obs_test_idx, None]])

ORACLE_CATE_ACTG = float(np.max(tau_full))

print(
    f"n_obs_train={len(obs_train_idx)}, n_obs_test={len(obs_test_idx)}, n_rct={len(rct_idx)}"
)
print(
    f"tau range: [{tau_full.min():.3f}, {tau_full.max():.3f}]  oracle CATE: {ORACLE_CATE_ACTG:.3f}"
)
print(f"y_obs_train range: [{y_obs_train.min():.3f}, {y_obs_train.max():.3f}]")

In [ ]:
def compute_pehe(predictor, X_test, mu0_test, mu1_test):
    """sqrt(PEHE) for any object with a .predict(X) method."""
    tau_true = mu1_test - mu0_test
    X_t = X_test.copy()
    X_t[:, TREATMENT_IDX] = 1.0
    X_c = X_test.copy()
    X_c[:, TREATMENT_IDX] = 0.0
    tau_hat = predictor.predict(X_t) - predictor.predict(X_c)
    return float(np.sqrt(np.mean((tau_hat - tau_true) ** 2)))

## 4. ACTG D_0 and pool construction

In [ ]:
def make_initial_dataset_actg(X_rct, y_rct, t_rct, tau_rct, n_init=80, seed=42):
    """Sample D_0 from RCT pool: 70/30 control/treated, biased toward low-CATE units."""
    rng = np.random.default_rng(seed)
    ctrl_idx = np.where(t_rct < 0.5)[0]
    trt_idx = np.where(t_rct >= 0.5)[0]
    n_ctrl = min(int(n_init * 0.70), len(ctrl_idx))
    n_trt = min(n_init - n_ctrl, len(trt_idx))
    low_thresh = np.percentile(tau_rct, 60)
    ctrl_low = ctrl_idx[tau_rct[ctrl_idx] <= low_thresh]
    trt_low = trt_idx[tau_rct[trt_idx] <= low_thresh]
    ctrl_pool = ctrl_low if len(ctrl_low) >= n_ctrl else ctrl_idx
    trt_pool = trt_low if len(trt_low) >= n_trt else trt_idx
    init_ctrl = rng.choice(ctrl_pool, n_ctrl, replace=False)
    init_trt = rng.choice(trt_pool, n_trt, replace=False)
    init_idx = np.concatenate([init_ctrl, init_trt])
    rng.shuffle(init_idx)
    return X_rct[init_idx], y_rct[init_idx], init_idx


t_rct = t_raw[rct_idx]
tau_rct = tau_full[rct_idx]

X0, y0, init_idx_rct = make_initial_dataset_actg(
    X_rct, y_rct, t_rct, tau_rct, n_init=80, seed=data_seed
)

pool_mask_base = np.ones(len(rct_idx), dtype=bool)
pool_mask_base[init_idx_rct] = False
pool_X_base = X_rct[pool_mask_base]
pool_y_base = y_rct[pool_mask_base]
pool_tau_base = tau_rct[pool_mask_base]
tau_0_base = tau_rct[init_idx_rct]

print(f"D_0:  n={len(X0)}, y [{y0.min():.3f}, {y0.max():.3f}], mean {y0.mean():.3f}")
print(f"D_0 max CATE: {tau_0_base.max():.3f}  |  Oracle CATE: {ORACLE_CATE_ACTG:.3f}")
print(f"Pool: {pool_mask_base.sum()} RCT units  |  Test: {len(obs_test_idx)} units")
T_col = X0[:, -1]
print(
    f"D_0 treatment split: {(T_col < 0.5).sum()} control / {(T_col >= 0.5).sum()} treated"
)
print(f"obs_train: {len(obs_train_idx)} patients (used only by R-Design Stage 1)")

## 5. MARICL-AL components - base models: XGBoost, TabPFN, TabICL

`train_tabpfn` and `train_tabicl` are drop-in wrappers with the same
`(X, y, seed) → model` signature as `train_xgb`.  `run_maricl_al` accepts
a `base_model_fn` argument so any of the three drives MARICL without
touching the residual-correction logic.

In [ ]:
def train_xgb(X, y, seed=0):
    m = xgb.XGBRegressor(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.08,
        reg_lambda=1.0,
        reg_alpha=0.1,
        subsample=0.9,
        random_state=seed,
        tree_method="hist",
        verbosity=0,
    )
    m.fit(np.asarray(X), np.asarray(y))
    return m


def train_tlearner(X, y, seed=0, treatment_idx=None, threshold=0.5):
    """T-learner: two separate XGBoost models split on binary treatment column."""
    if treatment_idx is None:
        treatment_idx = TREATMENT_IDX
    X = np.asarray(X)
    y = np.asarray(y)
    t = X[:, treatment_idx] >= threshold
    m0 = xgb.XGBRegressor(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.08,
        random_state=seed,
        tree_method="hist",
        verbosity=0,
    )
    m1 = xgb.XGBRegressor(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.08,
        random_state=seed,
        tree_method="hist",
        verbosity=0,
    )
    if t.sum() > 0:
        m1.fit(X[t], y[t])
    if (~t).sum() > 0:
        m0.fit(X[~t], y[~t])

    class TLearner:
        def predict(self, X):
            X = np.asarray(X)
            t = X[:, treatment_idx] >= threshold
            out = np.zeros(len(X))
            if t.sum() > 0:
                out[t] = m1.predict(X[t])
            if (~t).sum() > 0:
                out[~t] = m0.predict(X[~t])
            return out

    return TLearner()


def nn_distances(X_query, X_ref, standardise=True, exclude_self=False):
    X_query = np.atleast_2d(X_query)
    X_ref = np.atleast_2d(X_ref)
    if standardise:
        mu = X_ref.mean(0)
        sd = X_ref.std(0)
        sd[sd < 1e-9] = 1.0
        Xq = (X_query - mu) / sd
        Xr = (X_ref - mu) / sd
    else:
        Xq, Xr = X_query, X_ref
    d = np.linalg.norm(Xq[:, None, :] - Xr[None, :, :], axis=-1)
    if exclude_self:
        np.fill_diagonal(d, np.inf)
    return d.min(axis=1)


def epistemic_uncertainty(X_query, X_train, maricl, phi=0.6):
    dnn = nn_distances(X_query, X_train, standardise=True)
    med = max(
        float(
            np.median(
                nn_distances(X_train, X_train, standardise=True, exclude_self=True)
            )
        ),
        1e-6,
    )
    lam = math.log(2.0) / (med**2)
    u_ext = 1.0 - np.exp(-lam * dnn**2)
    _, deltas, _, _, _ = maricl.predict_components(X_query)
    if deltas.shape[1] >= 2:
        u_dis = deltas.std(axis=1)
        cap = max(np.percentile(u_dis, 95), 1e-9)
        u_dis = np.minimum(u_dis / cap, 1.0)
    else:
        u_dis = np.zeros(len(X_query))
    return phi * u_ext + (1.0 - phi) * u_dis, u_ext, u_dis


def mechanism_alignment(X_query, maricl):
    _, deltas, _, p_arr, bars = maricl.predict_components(X_query)
    if deltas.shape[1] == 0:
        return np.zeros(len(X_query))
    return (p_arr[None, :] * np.maximum(deltas - bars[None, :], 0.0)).sum(axis=1)


def acquisition(X_query, maricl, X_train, beta=0.4, mu=0.3, phi=0.6):
    y_hat = maricl.predict(X_query)
    u, _, _ = epistemic_uncertainty(X_query, X_train, maricl, phi=phi)
    m_x = mechanism_alignment(X_query, maricl)
    return y_hat + beta * u + mu * m_x, dict(y_hat=y_hat, u=u, m=m_x)


def greedy_diverse_batch(X_pool, acq_scores, B, X_train, eta=None):
    n = len(X_pool)
    if len(X_train) >= 2:
        d_tr = np.linalg.norm(X_train[:, None, :] - X_train[None, :, :], axis=-1)
        iu = np.triu_indices_from(d_tr, k=1)
        bandwidth = max(float(np.median(d_tr[iu])), 1e-6)
    else:
        bandwidth = 0.3

    def _select(eta_val):
        chosen, chosen_idx = [], []
        i0 = int(np.argmax(acq_scores))
        chosen.append(X_pool[i0])
        chosen_idx.append(i0)
        max_sim = np.zeros(n)
        for b in range(1, B):
            d2 = np.linalg.norm(X_pool - chosen[-1], axis=1) ** 2
            sim_new = np.exp(-d2 / (2 * bandwidth**2))
            max_sim = np.maximum(max_sim, sim_new)
            score = acq_scores - eta_val * max_sim
            score[chosen_idx] = -np.inf
            i = int(np.argmax(score))
            chosen.append(X_pool[i])
            chosen_idx.append(i)
        return np.asarray(chosen), chosen_idx, bandwidth

    if eta is not None:
        return _select(eta)
    target = 1.5 * bandwidth
    lo, hi = 0.0, 5.0
    for _ in range(8):
        mid = 0.5 * (lo + hi)
        batch, _, bw = _select(mid)
        d_b = np.linalg.norm(batch[:, None, :] - batch[None, :, :], axis=-1)
        iu = np.triu_indices_from(d_b, k=1)
        med_b = float(np.median(d_b[iu])) if len(iu[0]) else 0.0
        if med_b >= target:
            hi = mid
        else:
            lo = mid
    return _select(hi)


def template_similarity(t_prev, t_curr):
    if not t_prev or not t_curr:
        return 0.0
    e = embedder.embed_documents([t_prev, t_curr])
    a, b = np.asarray(e[0]), np.asarray(e[1])
    return float((a @ b) / max(np.linalg.norm(a) * np.linalg.norm(b), 1e-9))


def should_refit_maricl(
    round_t, prev_templates, cur_templates, theta_T=0.85, refit_every=3
):
    if round_t % refit_every == 0:
        return True, f"scheduled refit (every {refit_every})"
    sims = [template_similarity(p, c) for p, c in zip(prev_templates, cur_templates)]
    if any(s < theta_T for s in sims):
        return True, f'template drift (sims={[f"{s:.2f}" for s in sims]})'
    return False, f'templates stable (sims={[f"{s:.2f}" for s in sims]})'

In [ ]:
def train_causalpfn(X, y, seed=0, treatment_idx=None, threshold=0.5):
    """CausalPFN: estimates CATE then predicts Y0 + T*CATE.
    Requires: pip install causalpfn  (~2GB download)
    """
    if treatment_idx is None:
        treatment_idx = TREATMENT_IDX
    try:
        from causalpfn import CATEEstimator
        import causalpfn.causal_estimator as _ce
        import torch
        from sklearn.ensemble import GradientBoostingRegressor as _GBR

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    except ImportError:
        raise ImportError("Run: pip install causalpfn")

    class _SafeGBR(_GBR):
        def __init__(self, **kw):
            kw["min_samples_leaf"] = max(1, kw.get("min_samples_leaf", 1))
            super().__init__(**kw)

    _ce.GradientBoostingRegressor = _SafeGBR
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)
    T = (X[:, treatment_idx] >= threshold).astype(np.float32)
    X11 = X[:, :treatment_idx]
    estimator = CATEEstimator(device=device, verbose=False)
    estimator.fit(X11, T, y)
    ctrl = T < threshold
    bX = X11[ctrl] if ctrl.sum() > 0 else X11
    by = y[ctrl] if ctrl.sum() > 0 else y
    baseline = xgb.XGBRegressor(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.08,
        random_state=seed,
        tree_method="hist",
        verbosity=0,
    )
    baseline.fit(bX, by)

    class CausalPFNModel:
        def predict(self, X):
            X = np.asarray(X, dtype=np.float32)
            t = (X[:, treatment_idx] >= threshold).astype(float)
            x11 = X[:, :treatment_idx]
            y0 = baseline.predict(x11).astype(float)
            cate = estimator.estimate_cate(x11).astype(float)
            return y0 + t * cate

    return CausalPFNModel()


# Quick sanity check
assert (
    X0.shape[1] == TREATMENT_IDX + 1
), f"Expected {TREATMENT_IDX+1} cols, got {X0.shape[1]}"
model = train_tlearner(X0, y0, seed=0)
preds = model.predict(X0)
assert preds.shape == (len(X0),)
T_col = X0[:, TREATMENT_IDX]
print(
    f"T-learner sanity: n_treated={int((T_col>=0.5).sum())}, "
    f"pred range=[{preds.min():.3f}, {preds.max():.3f}]"
)

## 6. AL loop + baselines

`run_maricl_al` now accepts `base_model_fn` (default `train_xgb`) and `name`.
Pass `train_tabpfn` or `train_tabicl` to swap the surrogate - zero other changes.

In [ ]:
ACTG_FEATURES = [
    "age",
    "wtkg",
    "hemo",
    "homo",
    "drugs",
    "karnof",
    "race",
    "gender",
    "z30",
    "cd40",
    "cd80",
    "str2",
]
ACTG_DESCS = [
    "patient age in years at baseline (standardized)",
    "weight in kg at baseline (standardized)",
    "hemophilia indicator (binary: 1=yes)",
    "homosexual activity indicator (binary: 1=yes)",
    "history of IV drug use (binary: 1=yes)",
    "Karnofsky score 0-100 functional status (standardized)",
    "race (binary: 0=White, 1=non-white)",
    "gender (binary: 0=Female, 1=Male)",
    "ZDV use in 30 days prior to trial (binary)",
    "CD4 count at baseline (standardized)",
    "CD8 count at baseline (standardized)",
    "antiretroviral history (binary: 0=naive, 1=experienced)",
]
ACTG_DOMAIN = (
    "ACTG-175 HIV clinical trial (semi-synthetic). "
    "Binary treatment: T=0 ZDV monotherapy, T=1 combination/alternative therapy. "
    "Outcome: synthetic functional outcome combining MLP baseline with "
    "weight-Karnofsky interaction and age modulation (R-Design DGP). "
    "tau(x) = 1.5*sin(wtkg)*(karnof_hi+1) + 2*age. "
    "All continuous features standardized; binary features in {0,1}."
)
ACTG_FEATURES_T = ACTG_FEATURES + ["T"]
ACTG_DESCS_T = ACTG_DESCS + ["binary treatment (0=ZDV, 1=combination/other)"]


@dataclass
class ALResult:
    name: str
    X_hist: List[np.ndarray] = field(default_factory=list)
    y_hist: List[np.ndarray] = field(default_factory=list)
    best_so_far: List[float] = field(default_factory=list)
    pehe_per_round: List[float] = field(default_factory=list)
    templates_per_round: List[List[str]] = field(default_factory=list)


def run_maricl_al(
    X0,
    y0,
    pool_X,
    pool_y,
    pool_mu1=None,
    init_mu1=None,
    R=5,
    B=8,
    beta=0.4,
    mu_=0.3,
    phi=0.6,
    refit_every=3,
    theta_T=0.85,
    T_iter=3,
    seed=0,
    verbose=True,
    base_model_fn=None,
    name="MARICL-AL",
    X_test=None,
    mu0_test=None,
    mu1_test=None,
):
    """Trains base model on RCT data only (D_0 + accumulated acquired points).
    obs_train is deliberately excluded to avoid confounding bias contaminating
    the residuals that MARICL tries to correct.
    """
    if base_model_fn is None:
        base_model_fn = train_xgb
    X = X0.copy()
    y = y0.copy()
    pool_X = pool_X.copy()
    pool_y = pool_y.copy()
    pool_avail = np.ones(len(pool_X), dtype=bool)
    res = ALResult(name=name)
    res.X_hist.append(X.copy())
    res.y_hist.append(y.copy())
    _best0 = float(init_mu1.max()) if init_mu1 is not None else float(y.max())
    res.best_so_far.append(_best0)
    if verbose:
        print(f"\n=== D_0 ({len(y)} RCT examples, best tau={_best0:.3f}) ===")
    f_ml = base_model_fn(X, y, seed=seed)
    tau_fail = float(np.std(y) * 0.15)
    maricl = MARICL(
        K=num_hypothesis,
        kappa=0.3,
        T=T_iter,
        verbose=verbose,
        output_range=(-1.0, 1.0),
        tau_fail=tau_fail,
    )
    maricl.fit(X, y, f_ml, ACTG_FEATURES_T, ACTG_DESCS_T, ACTG_DOMAIN)
    prev_templates = [
        s.best_correction.T_k if s.best_correction else "(none)" for s in maricl.states_
    ]
    res.templates_per_round.append(prev_templates.copy())
    if X_test is not None:
        res.pehe_per_round.append(compute_pehe(maricl, X_test, mu0_test, mu1_test))
    for t in range(1, R + 1):
        if verbose:
            print(f"\n--- Round {t} ---")
        avail_idx = np.where(pool_avail)[0]
        if len(avail_idx) < B:
            if verbose:
                print(f"  Pool exhausted ({len(avail_idx)} left), stopping.")
            break
        pool_now = pool_X[avail_idx]
        a, info = acquisition(pool_now, maricl, X, beta=beta, mu=mu_, phi=phi)
        if verbose:
            print(
                f"  acq: y_hat[{info['y_hat'].min():.3f},{info['y_hat'].max():.3f}]  "
                f"u[{info['u'].min():.3f},{info['u'].max():.3f}]  "
                f"m[{info['m'].min():.3f},{info['m'].max():.3f}]"
            )
        batch_X, local_idx, bw = greedy_diverse_batch(
            pool_now, a, B=B, X_train=X, eta=None
        )
        global_idx = avail_idx[np.asarray(local_idx)]
        batch_y = pool_y[global_idx]
        batch_mu1 = pool_mu1[global_idx] if pool_mu1 is not None else None
        pool_avail[global_idx] = False
        if verbose:
            print(
                f"  batch B={B}: best y={batch_y.max():.3f}, mean={batch_y.mean():.3f}"
            )
        X = np.vstack([X, batch_X])
        y = np.concatenate([y, batch_y])
        res.X_hist.append(X.copy())
        res.y_hist.append(y.copy())
        if batch_mu1 is not None:
            res.best_so_far.append(
                float(max(res.best_so_far[-1], float(batch_mu1.max())))
            )
        else:
            res.best_so_far.append(float(y.max()))
        f_ml = base_model_fn(X, y, seed=seed)
        maricl.base_model = f_ml
        if t < R:
            if verbose:
                print("  checking MARICL refit gate ...")
            do_refit = t % refit_every == 0
            why = f"scheduled refit (every {refit_every})" if do_refit else ""
            if not do_refit:
                tf2 = float(np.std(y) * 0.15)
                tentative = MARICL(
                    K=num_hypothesis,
                    kappa=0.3,
                    T=T_iter,
                    verbose=False,
                    output_range=(-1.0, 1.0),
                    tau_fail=tf2,
                )
                tentative.fit(X, y, f_ml, ACTG_FEATURES_T, ACTG_DESCS_T, ACTG_DOMAIN)
                new_templates = [
                    s.best_correction.T_k if s.best_correction else "(none)"
                    for s in tentative.states_
                ]
                do_refit, why = should_refit_maricl(
                    t, prev_templates, new_templates, theta_T, refit_every
                )
                if do_refit:
                    maricl = tentative
                    prev_templates = new_templates
            else:
                tau_fail = float(np.std(y) * 0.15)
                maricl = MARICL(
                    K=num_hypothesis,
                    kappa=0.3,
                    T=T_iter,
                    verbose=verbose,
                    output_range=(-1.0, 1.0),
                    tau_fail=tau_fail,
                )
                maricl.fit(X, y, f_ml, ACTG_FEATURES_T, ACTG_DESCS_T, ACTG_DOMAIN)
                prev_templates = [
                    s.best_correction.T_k if s.best_correction else "(none)"
                    for s in maricl.states_
                ]
            if verbose:
                print(f"  -> {why}; refit={do_refit}")
        res.templates_per_round.append(prev_templates.copy())
        if X_test is not None:
            res.pehe_per_round.append(compute_pehe(maricl, X_test, mu0_test, mu1_test))
        if verbose:
            print(
                f"  best-so-far tau: {res.best_so_far[-1]:.3f}  "
                + (
                    f"sqrt(PEHE): {res.pehe_per_round[-1]:.4f}"
                    if res.pehe_per_round
                    else ""
                )
            )
    return res, maricl


def run_baseline(
    X0,
    y0,
    pool_X,
    pool_y,
    pool_mu1=None,
    init_mu1=None,
    policy="random",
    R=3,
    B=8,
    seed=0,
    verbose=False,
    gp_kappa=2.0,
    ei_xi=0.02,
    n_xgb_ens=7,
    X_test=None,
    mu0_test=None,
    mu1_test=None,
):
    """Trains on RCT data only. GPs on 1200+ obs points would be O(n³) = impractically slow."""
    LABELS = {
        "random": "Random",
        "exploit": "Exploit-only",
        "gp_ucb": "GP-UCB",
        "gp_ei": "GP-EI",
        "uncertainty": "Ens-sigma (XGB)",
    }
    if policy not in LABELS:
        raise ValueError(policy)
    label = LABELS[policy]
    rng = np.random.default_rng(seed + 10_000)
    X = X0.copy()
    y = y0.copy()
    pool_X = pool_X.copy()
    pool_y = pool_y.copy()
    pool_avail = np.ones(len(pool_X), dtype=bool)
    res = ALResult(name=label)
    res.X_hist.append(X.copy())
    res.y_hist.append(y.copy())
    _best0b = float(init_mu1.max()) if init_mu1 is not None else float(y0.max())
    res.best_so_far.append(_best0b)
    if X_test is not None:
        _m0 = train_tlearner(X, y, seed=seed)
        res.pehe_per_round.append(compute_pehe(_m0, X_test, mu0_test, mu1_test))

    def _fit_gp(Xtr, ytr, seed_gp):
        sc = StandardScaler()
        Xs = sc.fit_transform(np.asarray(Xtr, dtype=float))
        ys = np.asarray(ytr, dtype=float).ravel()
        kern = C(1.0, (1e-3, 1e3)) * RBF(
            length_scale=np.ones(Xs.shape[1]) * 0.35, length_scale_bounds=(0.03, 1.5)
        ) + WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-6, 0.06))
        gp = GaussianProcessRegressor(
            kernel=kern,
            normalize_y=True,
            alpha=1e-9,
            n_restarts_optimizer=1,
            random_state=seed_gp,
        )
        gp.fit(Xs, ys)
        return gp, sc

    for t in range(1, R + 1):
        avail_idx = np.where(pool_avail)[0]
        if len(avail_idx) < B:
            if verbose:
                print(f"  [{label}] pool exhausted at round {t}")
            break
        pool_now = np.asarray(pool_X[avail_idx], dtype=float)
        if policy == "random":
            local_idx = rng.choice(len(pool_now), B, replace=False)
        elif policy == "exploit":
            f_ml = train_xgb(X, y, seed=seed)
            local_idx = np.argsort(-f_ml.predict(pool_now))[:B]
        elif policy == "gp_ucb":
            gp, sc = _fit_gp(X, y, seed + 31_415 * t)
            mu_p, sigma = gp.predict(sc.transform(pool_now), return_std=True)
            local_idx = np.argsort(
                -(mu_p + gp_kappa * np.maximum(sigma.astype(float), 1e-9))
            )[:B]
        elif policy == "gp_ei":
            gp, sc = _fit_gp(X, y, seed + 27_182 * t)
            mu_p, sigma = gp.predict(sc.transform(pool_now), return_std=True)
            sigma = np.maximum(sigma.astype(float), 1e-9)
            improv = mu_p - float(np.max(y)) - ei_xi
            Z = improv / sigma
            ei = np.clip(improv * norm.cdf(Z) + sigma * norm.pdf(Z), 0, np.inf)
            local_idx = np.argsort(-ei)[:B]
        else:
            preds = [
                train_xgb(X, y, seed=seed + 997 * (j + 1) + 13 * t).predict(pool_now)
                for j in range(n_xgb_ens)
            ]
            local_idx = np.argsort(-np.std(np.stack(preds, axis=0), axis=0))[:B]
        global_idx = avail_idx[local_idx]
        batch_X = pool_now[local_idx]
        batch_y = pool_y[global_idx]
        batch_mu1b = pool_mu1[global_idx] if pool_mu1 is not None else None
        pool_avail[global_idx] = False
        X = np.vstack([X, batch_X])
        y = np.concatenate([y, batch_y])
        res.X_hist.append(X.copy())
        res.y_hist.append(y.copy())
        if batch_mu1b is not None:
            res.best_so_far.append(
                float(max(res.best_so_far[-1], float(batch_mu1b.max())))
            )
        else:
            res.best_so_far.append(float(y.max()))
        if X_test is not None:
            _mt = train_tlearner(X, y, seed=seed)
            res.pehe_per_round.append(compute_pehe(_mt, X_test, mu0_test, mu1_test))
        if verbose:
            print(
                f"  [{label}] round {t}: best-so-far {res.best_so_far[-1]:.3f}"
                + (
                    f"  sqrt(PEHE): {res.pehe_per_round[-1]:.4f}"
                    if res.pehe_per_round
                    else ""
                )
            )
    return res

In [ ]:
def run_rdesign(
    X0,
    y0,
    pool_X,
    pool_y,
    pool_mu1=None,
    init_mu1=None,
    R=5,
    B=8,
    seed=0,
    verbose=False,
    obs_model_fn=None,
    gp_kappa=2.0,
    X_test=None,
    mu0_test=None,
    mu1_test=None,
    X_obs=None,
    y_obs=None,
):
    """R-Design (Gao et al. 2026): TSR + residual GP-UCB.
    Stage 1 obs_model is trained on X_obs/y_obs (the large confounded observational set)
    and frozen throughout — this is the key R-Design advantage over pure-RCT methods.
    Stage 2 GP tracks residuals from acquired RCT points to debias the obs prior.
    """
    if obs_model_fn is None:
        obs_model_fn = train_xgb
    X = X0.copy()
    y = y0.copy()
    pool_X = pool_X.copy()
    pool_y = pool_y.copy()
    pool_avail = np.ones(len(pool_X), dtype=bool)
    res = ALResult(name="R-Design (TSR)")
    res.X_hist.append(X.copy())
    res.y_hist.append(y.copy())
    _best0r = float(init_mu1.max()) if init_mu1 is not None else float(y.max())
    res.best_so_far.append(_best0r)
    if X_test is not None:
        _m0r = train_tlearner(X, y, seed=seed)
        res.pehe_per_round.append(compute_pehe(_m0r, X_test, mu0_test, mu1_test))

    # Stage 1: train obs_model on observational data and FREEZE it
    if X_obs is not None:
        if verbose:
            print(
                f"  [R-Design] Stage 1: training obs model on {len(X_obs)} obs patients"
            )
        obs_model = obs_model_fn(X_obs, y_obs, seed=seed)
    else:
        if verbose:
            print("  [R-Design] Stage 1: no obs data — falling back to D_0")
        obs_model = obs_model_fn(X0, y0, seed=seed)

    X_exp = np.empty((0, X0.shape[1]))
    r_exp = np.empty(0)
    gp_res = sc_res = None

    def _fit_residual_gp(X_r, r, seed_gp):
        sc = StandardScaler()
        Xs = sc.fit_transform(np.asarray(X_r, dtype=float))
        rs = np.asarray(r, dtype=float).ravel()
        kern = C(1.0, (1e-3, 1e3)) * RBF(
            length_scale=np.ones(Xs.shape[1]) * 0.35, length_scale_bounds=(0.03, 1.5)
        ) + WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-6, 0.06))
        gp = GaussianProcessRegressor(
            kernel=kern,
            normalize_y=True,
            alpha=1e-9,
            n_restarts_optimizer=1,
            random_state=seed_gp,
        )
        gp.fit(Xs, rs)
        return gp, sc

    for t in range(1, R + 1):
        avail_idx = np.where(pool_avail)[0]
        if len(avail_idx) < B:
            break
        pool_now = np.asarray(pool_X[avail_idx], dtype=float)
        obs_pred = np.asarray(obs_model.predict(pool_now), dtype=float)
        if gp_res is not None:
            mu_r, sigma_r = gp_res.predict(sc_res.transform(pool_now), return_std=True)
            acq = (
                obs_pred
                + mu_r.astype(float)
                + gp_kappa * np.maximum(sigma_r.astype(float), 1e-9)
            )
        else:
            acq = obs_pred + gp_kappa * 0.05 * np.ones(len(pool_now))
        _, local_idx, _ = greedy_diverse_batch(pool_now, acq, B=B, X_train=X, eta=None)
        global_idx = avail_idx[np.asarray(local_idx)]
        batch_X = pool_now[np.asarray(local_idx)]
        batch_y = pool_y[global_idx]
        batch_mu1r = pool_mu1[global_idx] if pool_mu1 is not None else None
        pool_avail[global_idx] = False
        if verbose:
            print(f"  [R-Design] round {t}: best batch={batch_y.max():.3f}")
        batch_r = batch_y - np.asarray(obs_model.predict(batch_X), dtype=float)
        X_exp = np.vstack([X_exp, batch_X]) if len(X_exp) else batch_X.copy()
        r_exp = np.concatenate([r_exp, batch_r])
        if len(X_exp) >= 3:
            try:
                gp_res, sc_res = _fit_residual_gp(X_exp, r_exp, seed + 31415 * t)
            except Exception:
                pass
        if batch_mu1r is not None:
            res.best_so_far.append(
                float(max(res.best_so_far[-1], float(batch_mu1r.max())))
            )
        else:
            res.best_so_far.append(float(y.max()))
        X = np.vstack([X, batch_X])
        y = np.concatenate([y, batch_y])
        res.X_hist.append(X.copy())
        res.y_hist.append(y.copy())
        if X_test is not None:
            _mtr = train_tlearner(X, y, seed=seed)
            res.pehe_per_round.append(compute_pehe(_mtr, X_test, mu0_test, mu1_test))
    return res

## 7. Run all three MARICL-AL variants + classical baselines

One cell runs everything and prints the comparison table.
Set `SKIP_TABPFN` or `SKIP_TABICL` to `True` if those packages are not yet installed.

In [ ]:
# ── flags ──────────────────────────────────────────────────────────────────
SKIP_TABPFN = True
SKIP_TABICL = True
SKIP_TLEARNER = False
SKIP_CAUSALPFN = False

R = 5
B = 8
T_iter = 3

# MARICL-AL and classical baselines use RCT data only
KW = dict(
    R=R,
    B=B,
    pool_X=pool_X_base,
    pool_y=pool_y_base,
    pool_mu1=pool_tau_base,
    init_mu1=tau_0_base,
    T_iter=T_iter,
    beta=0.5,
    mu_=1.5,
    phi=0.6,
    refit_every=3,
    verbose=True,
    X_test=X_test_actg,
    mu0_test=mu0_test_actg,
    mu1_test=mu1_test_actg,
)

print("\n" + "=" * 60)
print("MARICL-AL  |  XGBoost")
print("=" * 60)
t0 = time.time()
res_xgb, maricl_xgb = run_maricl_al(
    X0, y0, **KW, base_model_fn=train_xgb, name="MARICL-AL (XGBoost)"
)
print(
    f'  done in {time.time()-t0:.0f}s  |  PEHE: {[f"{v:.4f}" for v in res_xgb.pehe_per_round]}'
)

if not SKIP_TLEARNER:
    print("\n" + "=" * 60)
    print("MARICL-AL  |  T-Learner")
    print("=" * 60)
    t0 = time.time()
    res_tlearner, maricl_tlearner = run_maricl_al(
        X0, y0, **KW, base_model_fn=train_tlearner, name="MARICL-AL (T-Learner)"
    )
    print(
        f'  done in {time.time()-t0:.0f}s  |  PEHE: {[f"{v:.4f}" for v in res_tlearner.pehe_per_round]}'
    )
else:
    res_tlearner = None
    print("  [T-Learner skipped]")

if not SKIP_CAUSALPFN:
    print("\n" + "=" * 60)
    print("MARICL-AL  |  CausalPFN")
    print("=" * 60)
    t0 = time.time()
    res_causalpfn, maricl_causalpfn = run_maricl_al(
        X0, y0, **KW, base_model_fn=train_causalpfn, name="MARICL-AL (CausalPFN)"
    )
    print(
        f'  done in {time.time()-t0:.0f}s  |  PEHE: {[f"{v:.4f}" for v in res_causalpfn.pehe_per_round]}'
    )
else:
    res_causalpfn = None
    print("  [CausalPFN skipped]")

if not SKIP_TABPFN:
    from tabpfn import TabPFNRegressor

    def train_tabpfn(X, y, seed=0):
        Xa, ya = np.asarray(X, dtype=float), np.asarray(y, dtype=float)
        if len(Xa) > 1000:
            idx = np.random.default_rng(seed).choice(len(Xa), 1000, replace=False)
            Xa, ya = Xa[idx], ya[idx]
        m = TabPFNRegressor(device="cpu", N_ensemble_configurations=4)
        m.fit(Xa, ya)
        return m

    res_tabpfn, maricl_tabpfn = run_maricl_al(
        X0, y0, **KW, base_model_fn=train_tabpfn, name="MARICL-AL (TabPFN)"
    )
else:
    res_tabpfn = None
    print("  [TabPFN skipped]")
res_tabicl = None

In [ ]:
print("=" * 60)
print("Classical baselines (RCT data only)")
print("=" * 60)
_bkw = dict(
    R=R,
    B=B,
    pool_X=pool_X_base,
    pool_y=pool_y_base,
    pool_mu1=pool_tau_base,
    init_mu1=tau_0_base,
    verbose=True,
    X_test=X_test_actg,
    mu0_test=mu0_test_actg,
    mu1_test=mu1_test_actg,
)
res_gp_ucb = run_baseline(X0, y0, policy="gp_ucb", **_bkw)
res_gp_ei = run_baseline(X0, y0, policy="gp_ei", **_bkw)
res_ens_unc = run_baseline(X0, y0, policy="uncertainty", **_bkw)
res_exploit = run_baseline(X0, y0, policy="exploit", **_bkw)
res_random = run_baseline(X0, y0, policy="random", **_bkw, seed=random_seed)

# R-Design: Stage 1 obs_model trained on obs_train (the correct R-Design setup)
# This is the ONLY method that gets to use obs_train — it's the whole point of R-Design
print("=" * 60)
print("R-Design (TSR) — Stage 1 on obs_train")
print("=" * 60)
res_rdesign = run_rdesign(
    X0,
    y0,
    pool_X_base,
    pool_y_base,
    pool_mu1=pool_tau_base,
    init_mu1=tau_0_base,
    R=R,
    B=B,
    verbose=True,
    obs_model_fn=train_xgb,
    gp_kappa=2.0,
    X_test=X_test_actg,
    mu0_test=mu0_test_actg,
    mu1_test=mu1_test_actg,
    X_obs=X_obs_train,
    y_obs=y_obs_train,
)
print(f'  PEHE trajectory: {[f"{v:.4f}" for v in res_rdesign.pehe_per_round]}')

ALL = [
    r
    for r in [
        res_xgb,
        res_tlearner,
        res_causalpfn,
        res_tabpfn,
        res_tabicl,
        res_gp_ucb,
        res_gp_ei,
        res_ens_unc,
        res_exploit,
        res_random,
        res_rdesign,
    ]
    if r is not None
]

print("=" * 60)
print("COMPARISON TABLE")
print("=" * 60)
W = max(len(r.name) for r in ALL)
hdr = f"{'policy':<{W}}  " + "".join(f"  r{i}" for i in range(R + 1)) + "   final PEHE"
print(hdr)
print("-" * len(hdr))
for r in ALL:
    if not r.pehe_per_round:
        continue
    vals = "  ".join(f"{v:.3f}" for v in r.pehe_per_round)
    final = r.pehe_per_round[-1]
    marker = (
        " <-- best"
        if final == min(rr.pehe_per_round[-1] for rr in ALL if rr.pehe_per_round)
        else ""
    )
    print(f"{r.name:<{W}}  {vals}   {final:.4f}{marker}")

## 8. Plot results


In [ ]:
_g = globals()
ALL = [
    r
    for r in [
        _g.get("res_xgb"),
        _g.get("res_tlearner"),
        _g.get("res_causalpfn") if not _g.get("SKIP_CAUSALPFN", True) else None,
        _g.get("res_tabpfn") if not _g.get("SKIP_TABPFN", True) else None,
        _g.get("res_gp_ucb"),
        _g.get("res_gp_ei"),
        _g.get("res_ens_unc"),
        _g.get("res_exploit"),
        _g.get("res_random"),
        _g.get("res_rdesign"),
    ]
    if r is not None
]
if not ALL:
    raise RuntimeError("No results — run cells above first.")
R = _g.get("R", len(ALL[0].pehe_per_round) - 1)

COLORS = {
    "MARICL-AL (XGBoost)": "tab:blue",
    "MARICL-AL (T-Learner)": "tab:red",
    "MARICL-AL (CausalPFN)": "tab:pink",
    "MARICL-AL (TabPFN)": "tab:cyan",
    "GP-UCB": "tab:green",
    "GP-EI": "tab:olive",
    "Ens-sigma (XGB)": "tab:purple",
    "Exploit-only": "tab:orange",
    "Random": "tab:gray",
    "R-Design (TSR)": "tab:brown",
}
STYLES = {
    "MARICL-AL (XGBoost)": dict(lw=2.5, ls="-", marker="o", zorder=5),
    "MARICL-AL (T-Learner)": dict(lw=2.5, ls="--", marker="D", zorder=5),
    "MARICL-AL (CausalPFN)": dict(lw=2.5, ls="-.", marker="P", zorder=5),
    "R-Design (TSR)": dict(lw=2.0, ls="-.", marker="s", zorder=4),
}

rounds = np.arange(R + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for r in ALL:
    if not r.pehe_per_round:
        continue
    kw = STYLES.get(r.name, dict(lw=1.4, ls=":", marker=".", zorder=3))
    axes[0].plot(
        rounds, r.pehe_per_round, color=COLORS.get(r.name, "k"), label=r.name, **kw
    )
axes[0].set_xlabel("AL round")
axes[0].set_ylabel("sqrt(PEHE) on obs-test set  (lower is better)")
axes[0].set_title(
    "MARICL-AL vs. baselines on ACTG-175\n(pool-based AL, B=8, 5 rounds, sqrt PEHE)"
)
axes[0].set_xticks(rounds)
axes[0].grid(alpha=0.3)
axes[0].legend(loc="upper right", fontsize=8)

names = [r.name for r in ALL if r.pehe_per_round]
finals = [r.pehe_per_round[-1] for r in ALL if r.pehe_per_round]
cols = [COLORS.get(n, "k") for n in names]
order = np.argsort(finals)[::-1]
axes[1].barh(
    [names[i] for i in order],
    [finals[i] for i in order],
    color=[cols[i] for i in order],
)
axes[1].set_xlabel("final sqrt(PEHE)  (lower = better)")
axes[1].set_title(f"Final sqrt(PEHE) after {R} rounds — ACTG-175")
axes[1].grid(alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

## 9. Inspect learned corrections


In [ ]:
print("Final corrections (XGBoost base):")
for k, (s, p) in enumerate(zip(maricl_xgb.states_, maricl_xgb.scores_)):
    print(f"\n--- correction {k}  (p_k={p:.3f}, best at iter {s.best_t}) ---")
    if s.best_correction:
        print(f"T_k: {s.best_correction.T_k}")
        print(f"f_k: {s.best_correction.f_k}")

In [ ]:
print("sqrt(PEHE) on obs-test set:")
print("-" * 40)
_g = globals()
for _name, _model in [
    ("MARICL-AL (XGBoost)", _g.get("maricl_xgb")),
    (
        "MARICL-AL (T-Learner)",
        _g.get("maricl_tlearner") if not _g.get("SKIP_TLEARNER", True) else None,
    ),
    (
        "MARICL-AL (CausalPFN)",
        _g.get("maricl_causalpfn") if not _g.get("SKIP_CAUSALPFN", True) else None,
    ),
]:
    if _model is None:
        continue
    pehe = compute_pehe(_model, X_test_actg, mu0_test_actg, mu1_test_actg)
    print(f"  {_name}: {pehe:.4f}")

# Acquisition surface: age vs wtkg slice
if _g.get("maricl_xgb") and _g.get("res_xgb"):
    Xall = res_xgb.X_hist[-1]
    median_x = np.median(Xall, axis=0)
    age_vals = np.linspace(Xall[:, 0].min(), Xall[:, 0].max(), 40)
    wtkg_vals = np.linspace(Xall[:, 1].min(), Xall[:, 1].max(), 40)
    xx, yy = np.meshgrid(age_vals, wtkg_vals)
    grid = np.tile(median_x, (xx.size, 1))
    grid[:, 0] = xx.ravel()  # age
    grid[:, 1] = yy.ravel()  # wtkg
    grid[:, TREATMENT_IDX] = 1.0

    y_hat = maricl_xgb.predict(grid).reshape(xx.shape)
    u, _, _ = epistemic_uncertainty(grid, Xall, maricl_xgb)
    u = u.reshape(xx.shape)
    m_x = mechanism_alignment(grid, maricl_xgb).reshape(xx.shape)
    a_surf = y_hat + 0.5 * u + 1.5 * m_x

    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    for ax, Z, title in zip(
        axes,
        [y_hat, u, m_x, a_surf],
        [
            "predicted outcome (T=1)",
            "uncertainty u(x)",
            "mech. alignment m(x)",
            "acquisition a(x)",
        ],
    ):
        im = ax.contourf(xx, yy, Z, 20, cmap="viridis")
        ax.scatter(
            Xall[: len(X0), 0],
            Xall[: len(X0), 1],
            c="white",
            edgecolor="k",
            s=14,
            label="D_0",
            lw=0.4,
        )
        if len(Xall) > len(X0):
            ax.scatter(
                Xall[len(X0) :, 0],
                Xall[len(X0) :, 1],
                c="red",
                edgecolor="k",
                s=20,
                marker="X",
                label="acquired",
                lw=0.4,
            )
        ax.set_xlabel("age (std)")
        ax.set_ylabel("wtkg (std)")
        ax.set_title(title)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    axes[0].legend(loc="upper left", fontsize=8)
    plt.suptitle("Acquisition slice: (age, wtkg), other features = median, T=1", y=1.02)
    plt.tight_layout()
    plt.show()